# tt-vae-gan: Timbre Transfer Training & Inference

> ⚠️ **DEPRECATED FOR ACTIVE TRAINING — KEPT FOR DEFENSE REFERENCE.**
> This notebook trains the **VAE-GAN POC** (Phase 4B, trumpet → violin). The active
> project has moved to a score-conditioned diffusion U-Net (see `train.py` /
> `colab/train_sanity.ipynb`). Kept unchanged for defense reproducibility.

**Music Style Transfer Project — Phase 4B**

This notebook trains a VAE-GAN timbre transfer model and runs end-to-end inference
through our complete pipeline:

```
WAV → Demucs → Mel (80-band, 22kHz) → [VAE-GAN] → Modified Mel → BigVGAN → WAV
```

**Architecture**: VAE-GAN (CycleGAN-style, unpaired)
- 1 shared Encoder → latent space
- N Generators (one per instrument)
- N Discriminators (PatchGAN)
- 6 losses: GAN + KL + Identity + KL' + Cycle + Latent L1
- ~2 GB VRAM (fits Colab free tier!)

**Based on**: [ebadawy/voice_conversion](https://github.com/ebadawy/voice_conversion) → [RussellSB/tt-vae-gan](https://github.com/RussellSB/tt-vae-gan)

**Authors**: Yotam & Gal | February 2026

---

### §37 cell-header convention

Every code cell in this notebook is preceded by a markdown header that
follows the project standard (ENGINEERING_DECISIONS §37, see
[colab/README.md](README.md)):

> **What this does.** plain-English summary.  
> **Inputs.** files / variables / env read.  
> **Outputs.** files / variables / state written.  
> **Action required.** anything the user must edit, click, approve, or run
> elsewhere. Prefix the heading with **⚠** when the action requires switching
> machines (e.g. F1 backfill needs `basic_pitch_env` locally).  
> **Runtime.** order-of-magnitude (seconds / minutes / hours).

Stub headers below carry `TODO` placeholders — fill them in when you next
edit the cell. `colab/postprocessing.ipynb` is the reference implementation.


## 0. GPU Check & Setup

<!-- §37-stub -->
## Cell 1 — TODO short title

**What this does.** TODO

**Inputs.** TODO

**Outputs.** TODO

**Action required.** TODO (or _none_)

**Runtime.** TODO (seconds / minutes / hours)


In [ ]:
!nvidia-smi
import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

/bin/bash: line 1: nvidia-smi: command not found

PyTorch: 2.9.0+cpu
CUDA available: False


## 1. Install Dependencies & Mount Drive

<!-- §37-stub -->
## Cell 2 — TODO short title

**What this does.** TODO

**Inputs.** TODO

**Outputs.** TODO

**Action required.** TODO (or _none_)

**Runtime.** TODO (seconds / minutes / hours)


In [ ]:
import os
import sys
import math
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import pickle
from tqdm.auto import tqdm

cuda = torch.cuda.is_available()
print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__} | CUDA: {cuda}")
if cuda:
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Model Architecture

The complete VAE-GAN model — Encoder, Generator, Discriminator — adapted for
our 80-mel pipeline.

Architecture dimensions at 80 mels:
- Encoder: (1, 80, 128) → Conv7 → 2× downsample → (256, 20, 32) → 4 ResBlocks → latent z
- Generator: z → shared ResBlock → 3 ResBlocks → 2× upsample → (1, 80, 128) → Tanh
- Discriminator: (1, 80, 128) → 4 conv blocks → (1, 5, 8) PatchGAN decisions

Note: 80 ÷ 16 = 5 (integer) ✓ — Discriminator works correctly with 80 mels.

<!-- §37-stub -->
## Cell 3 — TODO short title

**What this does.** TODO

**Inputs.** TODO

**Outputs.** TODO

**Action required.** TODO (or _none_)

**Runtime.** TODO (seconds / minutes / hours)


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Variable
import numpy as np


def weights_init_normal(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        torch.nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm2d') != -1:
        torch.nn.init.normal_(m.weight.data, 1.0, 0.02)
        torch.nn.init.constant_(m.bias.data, 0.0)


class LambdaLR:
    def __init__(self, n_epochs, offset, decay_start_epoch):
        assert (n_epochs - decay_start_epoch) > 0
        self.n_epochs = n_epochs
        self.offset = offset
        self.decay_start_epoch = decay_start_epoch

    def step(self, epoch):
        return 1.0 - max(0, epoch + self.offset - self.decay_start_epoch) / \
            (self.n_epochs - self.decay_start_epoch)


class ResidualBlock(nn.Module):
    def __init__(self, features):
        super().__init__()
        self.conv_block = nn.Sequential(
            nn.ReflectionPad2d(1),
            nn.Conv2d(features, features, 3),
            nn.InstanceNorm2d(features),
            nn.ReLU(inplace=True),
            nn.ReflectionPad2d(1),
            nn.Conv2d(features, features, 3),
            nn.InstanceNorm2d(features),
        )

    def forward(self, x):
        return x + self.conv_block(x)


class Encoder(nn.Module):
    def __init__(self, in_channels=1, dim=64, n_downsample=2):
        super().__init__()
        layers = [
            nn.ReflectionPad2d(3),
            nn.Conv2d(in_channels, dim, 7),
            nn.InstanceNorm2d(dim),
            nn.LeakyReLU(0.2, inplace=True),
        ]
        for _ in range(n_downsample):
            layers += [
                nn.Conv2d(dim, dim * 2, 4, stride=2, padding=1),
                nn.InstanceNorm2d(dim * 2),
                nn.ReLU(inplace=True),
            ]
            dim *= 2
        for _ in range(4):
            layers += [ResidualBlock(dim)]
        self.model_blocks = nn.Sequential(*layers)

    def reparameterization(self, mu):
        Tensor = torch.cuda.FloatTensor if mu.is_cuda else torch.FloatTensor
        z = Variable(Tensor(np.random.normal(0, 1, mu.shape)))
        return z + mu

    def forward(self, x):
        mu = self.model_blocks(x)
        z = self.reparameterization(mu)
        return mu, z


class Generator(nn.Module):
    def __init__(self, out_channels=1, dim=64, n_upsample=2, shared_block=None):
        super().__init__()
        self.shared_block = shared_block
        layers = []
        dim = dim * 2 ** n_upsample
        for _ in range(3):
            layers += [ResidualBlock(dim)]
        for _ in range(n_upsample):
            layers += [
                nn.ConvTranspose2d(dim, dim // 2, 4, stride=2, padding=1),
                nn.InstanceNorm2d(dim // 2),
                nn.LeakyReLU(0.2, inplace=True),
            ]
            dim = dim // 2
        layers += [nn.ReflectionPad2d(3), nn.Conv2d(dim, out_channels, 7), nn.Tanh()]
        self.model_blocks = nn.Sequential(*layers)

    def forward(self, x):
        x = self.shared_block(x)
        return self.model_blocks(x)


class Discriminator(nn.Module):
    def __init__(self, input_shape):
        super().__init__()
        channels, height, width = input_shape
        self.output_shape = (1, height // 2**4, width // 2**4)

        def block(in_f, out_f, normalize=True):
            layers = [nn.Conv2d(in_f, out_f, 4, stride=2, padding=1)]
            if normalize:
                layers.append(nn.InstanceNorm2d(out_f))
            layers.append(nn.LeakyReLU(0.2, inplace=True))
            return layers

        self.model = nn.Sequential(
            *block(channels, 64, normalize=False),
            *block(64, 128),
            *block(128, 256),
            *block(256, 512),
            nn.Conv2d(512, 1, 3, padding=1)
        )

    def forward(self, img):
        return self.model(img)


# Quick architecture test with our 80-mel config
IMG_HEIGHT = 80   # Our pipeline: 80 mel bins
IMG_WIDTH = 128   # 128 frames per training patch
CHANNELS = 1
DIM = 32          # Base filter count (original: 32)
N_DOWNSAMPLE = 2
N_SPKRS = 2

shared_dim = DIM * 2 ** N_DOWNSAMPLE
input_shape = (CHANNELS, IMG_HEIGHT, IMG_WIDTH)

enc = Encoder(dim=DIM, in_channels=CHANNELS, n_downsample=N_DOWNSAMPLE)
shared_G = ResidualBlock(features=shared_dim)
gen = Generator(dim=DIM, out_channels=CHANNELS, n_upsample=N_DOWNSAMPLE,
                shared_block=shared_G)
disc = Discriminator(input_shape)

# Test forward pass
x = torch.randn(2, CHANNELS, IMG_HEIGHT, IMG_WIDTH)
mu, z = enc(x)
out = gen(z)
d_out = disc(out)

print(f"Input:          {x.shape}")
print(f"Encoder latent: {mu.shape}")
print(f"Generator out:  {out.shape}")
print(f"Discriminator:  {d_out.shape}")
print(f"\nD output shape: {disc.output_shape}")
print(f"\n✓ Architecture verified for {IMG_HEIGHT}-mel pipeline!")

# Parameter count
total = sum(p.numel() for p in enc.parameters())
total += sum(p.numel() for p in gen.parameters())
total += sum(p.numel() for p in disc.parameters())
print(f"Total params (1 enc + 1 gen + 1 disc): {total:,}")
print(f"For {N_SPKRS} instruments: ~{total + (N_SPKRS-1) * sum(p.numel() for p in gen.parameters()) + (N_SPKRS-1) * sum(p.numel() for p in disc.parameters()):,}")

del enc, gen, disc, shared_G, x, mu, z, out, d_out
torch.cuda.empty_cache() if torch.cuda.is_available() else None

## 3. Mel Spectrogram Utilities

We use **our pipeline's mel params** (matching HiFi-GAN / BigVGAN vocoders).

| Param | Value | Reason |
|-------|-------|--------|
| sample_rate | 22050 | HiFi-GAN / BigVGAN training SR |
| n_mels | 80 | Vocoder expects 80 mel input |
| n_fft | 1024 | Vocoder STFT config |
| hop_length | 256 | Vocoder upsample ratio (8×8×2×2=256) |
| fmin / fmax | 0 / 8000 | Vocoder mel filterbank range |
| normalization | [0, 1] | Model internal (mapped from pipeline [-1,1]) |

<!-- §37-stub -->
## Cell 4 — TODO short title

**What this does.** TODO

**Inputs.** TODO

**Outputs.** TODO

**Action required.** TODO (or _none_)

**Runtime.** TODO (seconds / minutes / hours)


In [ ]:
import librosa
import soundfile as sf
from pathlib import Path
from scipy.signal import lfilter
import struct

# ─── Pipeline mel params (matches HiFi-GAN / BigVGAN vocoder) ──────────────
SAMPLE_RATE = 22050
N_FFT = 1024
NUM_MELS = 80
HOP_LENGTH = 256
WIN_LENGTH = 1024
FMIN = 0
FMAX = 8000
MIN_LEVEL_DB = -100
REF_LEVEL_DB = 20

# Number of frames per training patch
NUM_SAMPLES = 128  # ~1.5 seconds at hop=256, sr=22050

# VAD params
AUDIO_NORM_TARGET_DBFS = -30
PREEMPHASIS = 0.97


def load_wav(path):
    return librosa.load(path, sr=SAMPLE_RATE)[0]


def preprocess_wav(fpath):
    """Load and normalize volume."""
    wav, sr = librosa.load(str(fpath), sr=None)
    if sr != SAMPLE_RATE:
        wav = librosa.resample(wav, orig_sr=sr, target_sr=SAMPLE_RATE)
    # Volume normalization
    rms = np.mean(wav ** 2)
    if rms > 0:
        dBFS_change = AUDIO_NORM_TARGET_DBFS - 10 * np.log10(rms)
        if dBFS_change > 0:  # increase only
            wav = wav * (10 ** (dBFS_change / 20))
    return wav


def melspectrogram(y):
    """Compute normalized mel spectrogram in [0, 1] range."""
    D = librosa.stft(y=y, n_fft=N_FFT, hop_length=HOP_LENGTH, win_length=WIN_LENGTH)
    S = librosa.feature.melspectrogram(
        S=np.abs(D), sr=SAMPLE_RATE, n_fft=N_FFT,
        n_mels=NUM_MELS, fmin=FMIN, fmax=FMAX)
    S_db = 20 * np.log10(np.maximum(1e-5, S))
    # Normalize to [0, 1]
    S_norm = np.clip((S_db - MIN_LEVEL_DB) / -MIN_LEVEL_DB, 0, 1)
    return S_norm


def denormalize(S):
    """[0,1] → dB scale."""
    return (np.clip(S, 0, 1) * -MIN_LEVEL_DB) + MIN_LEVEL_DB


def reconstruct_waveform(mel, n_iter=32):
    """Griffin-Lim reconstruction from [0,1] mel."""
    S_db = denormalize(mel)
    S_amp = np.power(10.0, S_db * 0.05)
    S_linear = librosa.feature.inverse.mel_to_stft(
        S_amp, power=1, sr=SAMPLE_RATE, n_fft=N_FFT, fmin=FMIN, fmax=FMAX)
    wav = librosa.griffinlim(
        S_linear, n_iter=n_iter, hop_length=HOP_LENGTH, win_length=WIN_LENGTH)
    return wav


print(f"Mel params: SR={SAMPLE_RATE}, mels={NUM_MELS}, hop={HOP_LENGTH}, "
      f"n_fft={N_FFT}, fmin={FMIN}, fmax={FMAX}")
print(f"Training patch: {NUM_MELS} x {NUM_SAMPLES} = "
      f"{NUM_SAMPLES * HOP_LENGTH / SAMPLE_RATE:.2f}s per sample")

## 4. Data Preparation

**Option A**: Generate synthetic instrument data (smoke test — 2 min)

**Option B**: Upload URMP dataset instrument recordings to Drive

**Option C**: Upload your own instrument WAV files

Expected structure on Drive:
```
MusicProject_Colab/tt_vae_gan/data/data_urmp/
├── spkr_1/     (instrument 1 — e.g., trumpet WAV files)
│   ├── track1.wav
│   └── track2.wav
└── spkr_2/     (instrument 2 — e.g., violin WAV files)
    ├── track1.wav
    └── track2.wav
```

<!-- §37-stub -->
## Cell 5 — TODO short title

**What this does.** TODO

**Inputs.** TODO

**Outputs.** TODO

**Action required.** TODO (or _none_)

**Runtime.** TODO (seconds / minutes / hours)


In [ ]:
# ─── Configure dataset ───────────────────────────────────────────────────────
USE_SYNTHETIC = False        # False = use real URMP data from Drive
DATASET_NAME = 'data_synth' if USE_SYNTHETIC else 'data_urmp'
DATASET_PATH = f'{DRIVE_ROOT}/tt_vae_gan/data/{DATASET_NAME}'
N_SPKRS = 2                  # Number of instruments

if USE_SYNTHETIC:
    # ─── Option A: Generate synthetic instrument-like data ────────────────
    # Creates two "instruments" with different harmonic profiles
    # Just for testing the pipeline — replace with real data for quality results
    print("Generating synthetic instrument data for smoke test...")
    N_TRACKS = 10  # tracks per instrument
    TRACK_DURATION = 8  # seconds per track

    for spkr in range(1, N_SPKRS + 1):
        spkr_dir = f'{DATASET_PATH}/spkr_{spkr}'
        os.makedirs(spkr_dir, exist_ok=True)

        for t in range(N_TRACKS):
            duration = TRACK_DURATION
            n_samples = int(duration * SAMPLE_RATE)
            time_axis = np.arange(n_samples) / SAMPLE_RATE

            # Random base frequency (musical notes C3-C5)
            base_freq = 130.81 * (2 ** (np.random.randint(0, 24) / 12))

            if spkr == 1:
                # "Instrument 1" — bright, trumpet-like (strong odd harmonics)
                wav = (0.6 * np.sin(2 * np.pi * base_freq * time_axis)
                       + 0.3 * np.sin(2 * np.pi * 3 * base_freq * time_axis)
                       + 0.15 * np.sin(2 * np.pi * 5 * base_freq * time_axis)
                       + 0.08 * np.sin(2 * np.pi * 7 * base_freq * time_axis))
                # Add attack/decay envelope
                env = np.minimum(time_axis / 0.05, 1.0) * np.exp(-time_axis / 4)
            else:
                # "Instrument 2" — warm, violin-like (even + odd harmonics, vibrato)
                vibrato = 5 * np.sin(2 * np.pi * 5.5 * time_axis)
                freq = base_freq + vibrato
                phase = 2 * np.pi * np.cumsum(freq) / SAMPLE_RATE
                wav = (0.5 * np.sin(phase)
                       + 0.35 * np.sin(2 * phase)
                       + 0.2 * np.sin(3 * phase)
                       + 0.12 * np.sin(4 * phase)
                       + 0.06 * np.sin(5 * phase))
                env = np.minimum(time_axis / 0.15, 1.0) * np.exp(-time_axis / 6)

            wav = wav * env
            # Normalize
            wav = wav / (np.max(np.abs(wav)) + 1e-8) * 0.8
            # Add slight noise
            wav += np.random.randn(n_samples) * 0.005

            sf.write(f'{spkr_dir}/track_{t:03d}.wav', wav.astype(np.float32), SAMPLE_RATE)

        print(f"  spkr_{spkr}: {N_TRACKS} synthetic tracks ({TRACK_DURATION}s each)")

    print(f"\n✓ Synthetic data ready at {DATASET_PATH}")
    print("  (Replace with real instrument WAVs for meaningful results)")

else:
    # ─── Option B/C: Check uploaded data ──────────────────────────────────
    for spkr in range(1, N_SPKRS + 1):
        spkr_dir = f'{DATASET_PATH}/spkr_{spkr}'
        if os.path.exists(spkr_dir):
            wavs = [f for f in os.listdir(spkr_dir) if f.endswith('.wav')]
            print(f"spkr_{spkr}: {len(wavs)} WAV files")
            if wavs:
                y, sr = librosa.load(os.path.join(spkr_dir, wavs[0]), sr=None)
                print(f"  Example: {wavs[0]} ({len(y)/sr:.1f}s @ {sr}Hz)")
        else:
            print(f"⚠ spkr_{spkr}: Directory not found at {spkr_dir}")
            print(f"  Upload WAV files to: {spkr_dir}")

## 5. Preprocess: WAV → Mel Spectrograms

<!-- §37-stub -->
## Cell 6 — TODO short title

**What this does.** TODO

**Inputs.** TODO

**Outputs.** TODO

**Action required.** TODO (or _none_)

**Runtime.** TODO (seconds / minutes / hours)


In [ ]:
import random


class DataProc(torch.utils.data.Dataset):
    def __init__(self, dataset_path, split, n_spkrs):
        self.data_dict = pickle.load(
            open(f'{dataset_path}/data_{split}.pickle', 'rb'))
        self.n_spkrs = n_spkrs

    def __len__(self):
        total_len = 0
        for i in range(self.n_spkrs):
            tmp = np.sum([j.shape[1] for j in self.data_dict[i]])
            total_len = max(total_len, tmp / NUM_SAMPLES)
        return int(total_len)

    def __getitem__(self, item):
        samples = {}
        for i in range(self.n_spkrs):
            lens = [j.shape[1] for j in self.data_dict[i]]
            idx = np.random.choice(len(lens), p=np.array(lens) / sum(lens))
            data = self.data_dict[i][idx]
            rand_start = random.randint(0, data.shape[1] - NUM_SAMPLES)
            patch = np.array([data[:, rand_start:rand_start + NUM_SAMPLES]])
            # Rescale [0, 1] → [-1, 1] to match Generator's Tanh output range
            samples[i] = patch * 2.0 - 1.0
        return samples


# Test DataLoader
ds = DataProc(DATASET_PATH, 'train', N_SPKRS)
print(f"Dataset length: {len(ds)}")
sample = ds[0]
for k, v in sample.items():
    print(f"  spkr_{k+1}: {v.shape} (min={v.min():.3f}, max={v.max():.3f})")
print("\n(Data rescaled to [-1,1] to match Generator Tanh output)")

## 6. Dataset & DataLoader

<!-- §37-stub -->
## Cell 7 — TODO short title

**What this does.** TODO

**Inputs.** TODO

**Outputs.** TODO

**Action required.** TODO (or _none_)

**Runtime.** TODO (seconds / minutes / hours)


In [ ]:
import random
import shutil
import time as _time

# ─── Copy data to local SSD for faster I/O ──────────────────────────────────
# Drive reads go over network (~50-100 MB/s). Local /content/ is SSD (~500+ MB/s).
# This is a one-time copy at startup — data stays in local SSD for the session.
LOCAL_DATA_PATH = f'/content/data_local/{DATASET_NAME}'
if not os.path.exists(LOCAL_DATA_PATH):
    print(f"Copying dataset from Drive to local SSD...")
    t0 = _time.time()
    shutil.copytree(DATASET_PATH, LOCAL_DATA_PATH)
    print(f"  Done in {_time.time()-t0:.1f}s")
else:
    print(f"Local copy already exists at {LOCAL_DATA_PATH}")

# List local files
for f in sorted(os.listdir(LOCAL_DATA_PATH)):
    fp = os.path.join(LOCAL_DATA_PATH, f)
    if os.path.isfile(fp):
        print(f"  {f} ({os.path.getsize(fp)/1e6:.1f} MB)")

# Use local path for DataLoader (Drive path still used for checkpoints)
TRAIN_DATA_PATH = LOCAL_DATA_PATH


class DataProc(torch.utils.data.Dataset):
    def __init__(self, dataset_path, split, n_spkrs):
        self.data_dict = pickle.load(
            open(f'{dataset_path}/data_{split}.pickle', 'rb'))
        self.n_spkrs = n_spkrs

    def __len__(self):
        total_len = 0
        for i in range(self.n_spkrs):
            tmp = np.sum([j.shape[1] for j in self.data_dict[i]])
            total_len = max(total_len, tmp / NUM_SAMPLES)
        return int(total_len)

    def __getitem__(self, item):
        samples = {}
        for i in range(self.n_spkrs):
            lens = [j.shape[1] for j in self.data_dict[i]]
            idx = np.random.choice(len(lens), p=np.array(lens) / sum(lens))
            data = self.data_dict[i][idx]
            rand_start = random.randint(0, data.shape[1] - NUM_SAMPLES)
            patch = np.array([data[:, rand_start:rand_start + NUM_SAMPLES]])
            # Rescale [0, 1] → [-1, 1] to match Generator's Tanh output range
            samples[i] = patch * 2.0 - 1.0
        return samples


# Test DataLoader
ds = DataProc(TRAIN_DATA_PATH, 'train', N_SPKRS)
print(f"\nDataset length: {len(ds)}")
sample = ds[0]
for k, v in sample.items():
    print(f"  spkr_{k+1}: {v.shape} (min={v.min():.3f}, max={v.max():.3f})")
print("\n(Data rescaled to [-1,1] to match Generator Tanh output)")
print("(Data loaded from local SSD for faster training)")


## 7. Training

CycleGAN-style training with 6 loss components:

| Loss | Weight | Purpose |
|------|--------|---------|
| GAN (MSE) | λ₀=10 | Fool discriminator |
| KL | λ₁=0.1 | Regularize latent space |
| Identity (L1) | λ₂=100 | Preserve content when no transfer needed |
| KL' | λ₃=0.1 | Regularize translated latent |
| Cycle (L1) | λ₄=100 | Ensure reversibility A→B→A |
| Latent (L1) | λ₅=10 | Align latent representations |

<!-- §37-stub -->
## Cell 8 — TODO short title

**What this does.** TODO

**Inputs.** TODO

**Outputs.** TODO

**Action required.** TODO (or _none_)

**Runtime.** TODO (seconds / minutes / hours)


In [ ]:
import itertools
from IPython.display import clear_output
import matplotlib.pyplot as plt

# ─── Training hyperparameters ────────────────────────────────────────────────
# For smoke test with synthetic data: use 20 epochs (~2 min)
# For real training: set N_EPOCHS = 500
N_EPOCHS = 20 if USE_SYNTHETIC else 500
BATCH_SIZE = 16          # Increased from 4 to better utilize GPU
LR = 0.0001             # Adam learning rate
B1, B2 = 0.5, 0.999     # Adam betas
DECAY_EPOCH = N_EPOCHS // 2  # Start LR decay halfway
CHECKPOINT_INTERVAL = 10 if USE_SYNTHETIC else 50
RESUME_EPOCH = 500       # Training complete. Loads epoch 500 checkpoint for inference

MODEL_NAME = 'smoke_test' if USE_SYNTHETIC else 'pipeline_urmp'
SAVE_DIR = f'{DRIVE_ROOT}/tt_vae_gan/checkpoints/{MODEL_NAME}'
os.makedirs(SAVE_DIR, exist_ok=True)


def compute_kl(mu):
    return torch.mean(torch.pow(mu, 2))


# ─── Initialize models ───────────────────────────────────────────────────────
cuda = torch.cuda.is_available()
Tensor = torch.cuda.FloatTensor if cuda else torch.Tensor

shared_dim = DIM * 2 ** N_DOWNSAMPLE
input_shape = (CHANNELS, IMG_HEIGHT, IMG_WIDTH)

encoder = Encoder(dim=DIM, in_channels=CHANNELS, n_downsample=N_DOWNSAMPLE)
shared_G = ResidualBlock(features=shared_dim)
G = [Generator(dim=DIM, out_channels=CHANNELS, n_upsample=N_DOWNSAMPLE,
               shared_block=shared_G) for _ in range(N_SPKRS)]
D = [Discriminator(input_shape) for _ in range(N_SPKRS)]

if cuda:
    encoder = encoder.cuda()
    G = [g.cuda() for g in G]
    D = [d.cuda() for d in D]

# ─── Load or init weights ────────────────────────────────────────────────────
# Checkpoint files are named encoder_{epoch:02d}.pth (same epoch number as saved)
if RESUME_EPOCH > 0:
    encoder.load_state_dict(torch.load(f'{SAVE_DIR}/encoder_{RESUME_EPOCH:02d}.pth'))
    for n in range(N_SPKRS):
        G[n].load_state_dict(torch.load(f'{SAVE_DIR}/G{n+1}_{RESUME_EPOCH:02d}.pth'))
        D[n].load_state_dict(torch.load(f'{SAVE_DIR}/D{n+1}_{RESUME_EPOCH:02d}.pth'))
    print(f"Resumed from epoch {RESUME_EPOCH}")
else:
    encoder.apply(weights_init_normal)
    for n in range(N_SPKRS):
        G[n].apply(weights_init_normal)
        D[n].apply(weights_init_normal)
    print("Initialized with random weights")

# ─── Loss weights ────────────────────────────────────────────────────────────
lambda_0 = 10    # GAN
lambda_1 = 0.1   # KL
lambda_2 = 100   # Identity
lambda_3 = 0.1   # KL translated
lambda_4 = 100   # Cycle
lambda_5 = 10    # Latent

criterion_GAN = torch.nn.MSELoss()
criterion_pixel = torch.nn.L1Loss()
if cuda:
    criterion_GAN.cuda()
    criterion_pixel.cuda()

# ─── Optimizers ──────────────────────────────────────────────────────────────
optimizer_G = torch.optim.Adam(
    itertools.chain(encoder.parameters(), *[g.parameters() for g in G]),
    lr=LR, betas=(B1, B2))
optimizer_D = [torch.optim.Adam(d.parameters(), lr=LR, betas=(B1, B2)) for d in D]

lr_scheduler_G = torch.optim.lr_scheduler.LambdaLR(
    optimizer_G, lr_lambda=LambdaLR(N_EPOCHS, RESUME_EPOCH, DECAY_EPOCH).step)
lr_scheduler_D = [
    torch.optim.lr_scheduler.LambdaLR(
        opt_d, lr_lambda=LambdaLR(N_EPOCHS, RESUME_EPOCH, DECAY_EPOCH).step)
    for opt_d in optimizer_D]

# ─── DataLoader ──────────────────────────────────────────────────────────────
# num_workers=0: data is already in RAM (pickle loaded at init), so
# multiprocessing overhead > benefit. Eliminates subprocess pickling cost.
dataloader = torch.utils.data.DataLoader(
    DataProc(TRAIN_DATA_PATH, 'train', N_SPKRS),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)

print(f"\nReady to train:")
print(f"  Model: {MODEL_NAME}")
print(f"  Epochs: {RESUME_EPOCH} → {N_EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  LR: {LR} (decay from epoch {DECAY_EPOCH})")
print(f"  Instruments: {N_SPKRS}")
print(f"  Mel config: {IMG_HEIGHT}x{IMG_WIDTH}")
print(f"  Device: {'CUDA' if cuda else 'CPU'}")
print(f"  Checkpoints: {SAVE_DIR}")
if USE_SYNTHETIC:
    print(f"\n  ⚡ SMOKE TEST MODE: {N_EPOCHS} epochs (quick pipeline validation)")

# ─── Save training config to Drive ──────────────────────────────────────────
import json, datetime
training_config = {
    'model_name': MODEL_NAME,
    'dataset': DATASET_NAME,
    'n_epochs': N_EPOCHS,
    'batch_size': BATCH_SIZE,
    'lr': LR,
    'betas': [B1, B2],
    'decay_epoch': DECAY_EPOCH,
    'resume_epoch': RESUME_EPOCH,
    'checkpoint_interval': CHECKPOINT_INTERVAL,
    'n_spkrs': N_SPKRS,
    'mel_config': {
        'n_mels': IMG_HEIGHT,
        'segment_frames': IMG_WIDTH,
        'sample_rate': SAMPLE_RATE,
        'hop_length': HOP_LENGTH,
        'n_fft': N_FFT,
        'fmin': FMIN,
        'fmax': FMAX,
    },
    'architecture': {
        'dim': DIM,
        'n_downsample': N_DOWNSAMPLE,
        'channels': CHANNELS,
        'n_res_blocks': 3,  # hardcoded in Encoder/Generator classes
    },
    'loss_weights': {
        'gan': lambda_0,
        'kl': lambda_1,
        'identity': lambda_2,
        'kl_translated': lambda_3,
        'cycle': lambda_4,
        'latent': lambda_5,
    },
    'device': 'cuda' if cuda else 'cpu',
    'use_synthetic': USE_SYNTHETIC,
    'started_at': datetime.datetime.now().isoformat(),
}
with open(f'{SAVE_DIR}/training_config.json', 'w') as f:
    json.dump(training_config, f, indent=2)
print(f"\n✓ Training config saved to {SAVE_DIR}/training_config.json")

<!-- §37-stub -->
## Cell 9 — TODO short title

**What this does.** TODO

**Inputs.** TODO

**Outputs.** TODO

**Action required.** TODO (or _none_)

**Runtime.** TODO (seconds / minutes / hours)


In [ ]:
# ─── Training loop ──────────────────────────────────────────────────────────
# Guard: preserve history across cell re-runs (e.g. after Colab disconnect)
if 'history' not in dir() or not isinstance(history, dict):
    history = {'G': [], 'D': []}
GRAD_CLIP = 1.0  # Gradient clipping for training stability

for epoch in range(RESUME_EPOCH + 1, N_EPOCHS + 1):
    epoch_losses = {'G': [], 'D': []}

    for i, batch in enumerate(dataloader):
        for trg_id in range(N_SPKRS):
            src_ids = [j for j in range(N_SPKRS) if j != trg_id]
            src_id = random.choice(src_ids)

            X1 = Variable(batch[src_id].type(Tensor))
            X2 = Variable(batch[trg_id].type(Tensor))

            valid = Variable(Tensor(np.ones((X1.size(0), *D[trg_id].output_shape))),
                             requires_grad=False)
            fake = Variable(Tensor(np.zeros((X1.size(0), *D[trg_id].output_shape))),
                            requires_grad=False)

            # ─── Train Encoder + Generators ──────────────────────────────
            optimizer_G.zero_grad()

            mu1, Z1 = encoder(X1)
            mu2, Z2 = encoder(X2)

            feat_1 = mu1.view(mu1.size(0), -1).mean(dim=0)
            feat_2 = mu2.view(mu2.size(0), -1).mean(dim=0)

            recon_X1 = G[src_id](Z1)
            recon_X2 = G[trg_id](Z2)
            fake_X1 = G[src_id](Z2)
            fake_X2 = G[trg_id](Z1)

            mu1_, Z1_ = encoder(fake_X1)
            mu2_, Z2_ = encoder(fake_X2)
            cycle_X1 = G[src_id](Z2_)
            cycle_X2 = G[trg_id](Z1_)

            loss_G = (
                lambda_0 * criterion_GAN(D[src_id](fake_X1), valid)
                + lambda_0 * criterion_GAN(D[trg_id](fake_X2), valid)
                + lambda_1 * compute_kl(mu1)
                + lambda_1 * compute_kl(mu2)
                + lambda_2 * criterion_pixel(recon_X1, X1)
                + lambda_2 * criterion_pixel(recon_X2, X2)
                + lambda_3 * compute_kl(mu1_)
                + lambda_3 * compute_kl(mu2_)
                + lambda_4 * criterion_pixel(cycle_X1, X1)
                + lambda_4 * criterion_pixel(cycle_X2, X2)
                + lambda_5 * criterion_pixel(feat_1, feat_2)
            )

            loss_G.backward()
            # Gradient clipping to prevent explosion
            torch.nn.utils.clip_grad_norm_(encoder.parameters(), GRAD_CLIP)
            for g in G:
                torch.nn.utils.clip_grad_norm_(g.parameters(), GRAD_CLIP)
            optimizer_G.step()

            # ─── Train Discriminators ────────────────────────────────────
            optimizer_D[src_id].zero_grad()
            loss_D1 = (criterion_GAN(D[src_id](X1), valid)
                       + criterion_GAN(D[src_id](fake_X1.detach()), fake))
            loss_D1.backward()
            torch.nn.utils.clip_grad_norm_(D[src_id].parameters(), GRAD_CLIP)
            optimizer_D[src_id].step()

            optimizer_D[trg_id].zero_grad()
            loss_D2 = (criterion_GAN(D[trg_id](X2), valid)
                       + criterion_GAN(D[trg_id](fake_X2.detach()), fake))
            loss_D2.backward()
            torch.nn.utils.clip_grad_norm_(D[trg_id].parameters(), GRAD_CLIP)
            optimizer_D[trg_id].step()

            epoch_losses['G'].append(loss_G.item())
            epoch_losses['D'].append((loss_D1 + loss_D2).item())

    # ─── Per-epoch LR update (NOT per-batch!) ────────────────────────────
    lr_scheduler_G.step()
    for sched in lr_scheduler_D:
        sched.step()

    # ─── Epoch summary ───────────────────────────────────────────────────
    avg_G = np.mean(epoch_losses['G'])
    avg_D = np.mean(epoch_losses['D'])
    history['G'].append(avg_G)
    history['D'].append(avg_D)

    if epoch % 10 == 0 or epoch == RESUME_EPOCH + 1:
        remaining = (N_EPOCHS - epoch) * 2 / 60  # ~2 min/epoch estimate
        print(f"[Epoch {epoch}/{N_EPOCHS}] G: {avg_G:.4f} | D: {avg_D:.4f} | ~{remaining:.1f}h left")

    # ─── Save checkpoint ─────────────────────────────────────────────────
    if epoch % CHECKPOINT_INTERVAL == 0:
        torch.save(encoder.state_dict(), f'{SAVE_DIR}/encoder_{epoch:02d}.pth')
        for n in range(N_SPKRS):
            torch.save(G[n].state_dict(), f'{SAVE_DIR}/G{n+1}_{epoch:02d}.pth')
            torch.save(D[n].state_dict(), f'{SAVE_DIR}/D{n+1}_{epoch:02d}.pth')
        print(f"  → Checkpoint saved to Drive (epoch {epoch})")

# Final save
torch.save(encoder.state_dict(), f'{SAVE_DIR}/encoder_{epoch:02d}.pth')
for n in range(N_SPKRS):
    torch.save(G[n].state_dict(), f'{SAVE_DIR}/G{n+1}_{epoch:02d}.pth')
    torch.save(D[n].state_dict(), f'{SAVE_DIR}/D{n+1}_{epoch:02d}.pth')
print(f"\n✓ Training complete! Final checkpoint: epoch {epoch}")

# ─── Save loss history to Drive ─────────────────────────────────────────────
loss_data = {
    'epochs': list(range(RESUME_EPOCH + 1, RESUME_EPOCH + 1 + len(history['G']))),
    'generator_loss': history['G'],
    'discriminator_loss': history['D'],
}
with open(f'{SAVE_DIR}/loss_history.json', 'w') as f:
    json.dump(loss_data, f, indent=2)
print(f"✓ Loss history saved to {SAVE_DIR}/loss_history.json")

## 8. Training Loss Curves

<!-- §37-stub -->
## Cell 10 — TODO short title

**What this does.** TODO

**Inputs.** TODO

**Outputs.** TODO

**Action required.** TODO (or _none_)

**Runtime.** TODO (seconds / minutes / hours)


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history['G'], label='Generator', alpha=0.8)
ax1.set_title('Generator Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history['D'], label='Discriminator', color='orange', alpha=0.8)
ax2.set_title('Discriminator Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle(f'VAE-GAN Training — {MODEL_NAME}', fontsize=14)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/training_loss.png', dpi=150)
plt.show()
print(f"Final G loss: {history['G'][-1]:.4f}")
print(f"Final D loss: {history['D'][-1]:.4f}")

## 9. Inference — Timbre Transfer

Sliding-window inference with 4× overlap averaging.

Input: any mono WAV → mel spectrogram → VAE-GAN → transferred mel → Griffin-Lim → WAV

<!-- §37-stub -->
## Cell 11 — TODO short title

**What this does.** TODO

**Inputs.** TODO

**Outputs.** TODO

**Action required.** TODO (or _none_)

**Runtime.** TODO (seconds / minutes / hours)


In [ ]:
# ─── Inference setup ─────────────────────────────────────────────────────────
INFER_EPOCH = RESUME_EPOCH   # Use the loaded checkpoint epoch
TRG_ID = 1                   # 0-indexed: 0=trumpet (G1), 1=violin (G2)

spkr1_dir = f'{DATASET_PATH}/spkr_1'
available_wavs = sorted([f for f in os.listdir(spkr1_dir) if f.endswith('.wav')])
INPUT_WAV = os.path.join(spkr1_dir, available_wavs[0])
print(f"Input: {INPUT_WAV}")

N_OVERLAP = 4
OUTPUT_DIR = f'{DRIVE_ROOT}/tt_vae_gan/outputs/{MODEL_NAME}_ep{INFER_EPOCH}_G{TRG_ID+1}'
os.makedirs(OUTPUT_DIR, exist_ok=True)


def sliding_window_inference(wav_path):
    """Full audio timbre transfer with sliding window."""
    sample = preprocess_wav(wav_path)
    spect_src = melspectrogram(sample)  # [0, 1] range

    # Pad for consistent overlap
    spect_src = np.pad(spect_src, ((0, 0), (IMG_WIDTH, IMG_WIDTH)), 'constant')
    spect_trg = np.zeros_like(spect_src)

    length = spect_src.shape[1]
    hop = IMG_WIDTH // N_OVERLAP

    with torch.no_grad():
        for i in tqdm(range(0, length, hop), desc='Inference'):
            x = i + IMG_WIDTH
            if x <= length:
                S = spect_src[:, i:x]
            else:
                S = spect_src[:, i:]
                S = np.pad(S, ((0, 0), (0, x - length)), 'constant')

            S_tensor = torch.from_numpy(S).float().view(1, CHANNELS, IMG_HEIGHT, IMG_WIDTH)
            # Rescale [0,1] → [-1,1] to match training data range
            S_tensor = S_tensor * 2.0 - 1.0
            if cuda:
                S_tensor = S_tensor.cuda()

            mu, Z = encoder(S_tensor)
            fake = G[TRG_ID](Z)
            T = fake.squeeze().cpu().numpy()
            # Rescale Tanh output [-1,1] → [0,1]
            T = (T + 1.0) / 2.0

            for j in range(0, IMG_WIDTH, hop):
                y = j + hop
                if i + y > length:
                    break
                spect_trg[:, i+j:i+y] += T[:, j:y] / N_OVERLAP

    # Remove padding
    spect_src = spect_src[:, IMG_WIDTH:-IMG_WIDTH]
    spect_trg = spect_trg[:, IMG_WIDTH:-IMG_WIDTH]

    return spect_src, np.clip(spect_trg, 0, 1), sample


spect_src, spect_trg, original_wav = sliding_window_inference(INPUT_WAV)

# Reconstruct with Griffin-Lim
print('Reconstructing with Griffin-Lim...')
wav_transferred = reconstruct_waveform(spect_trg)

# Save
fname = Path(INPUT_WAV).stem
sf.write(f'{OUTPUT_DIR}/{fname}_original.wav', original_wav, SAMPLE_RATE)
sf.write(f'{OUTPUT_DIR}/{fname}_transferred_griffinlim.wav', wav_transferred, SAMPLE_RATE)
np.save(f'{OUTPUT_DIR}/{fname}_mel_src.npy', spect_src)
np.save(f'{OUTPUT_DIR}/{fname}_mel_trg.npy', spect_trg)

print(f'\n✓ Saved to {OUTPUT_DIR}/')
print(f'  Original: {fname}_original.wav')
print(f'  Transferred (Griffin-Lim): {fname}_transferred_griffinlim.wav')
print(f'  Mel arrays: {fname}_mel_src.npy, {fname}_mel_trg.npy')
if USE_SYNTHETIC:
    print(f'\n  Note: This is synthetic data — audio quality is not meaningful.')
    print(f'  The test validates that the full pipeline runs end-to-end. ✓')

<!-- §37-stub -->
## Cell 12 — TODO short title

**What this does.** TODO

**Inputs.** TODO

**Outputs.** TODO

**Action required.** TODO (or _none_)

**Runtime.** TODO (seconds / minutes / hours)


In [ ]:
# ─── Run inference ───────────────────────────────────────────────────────────
# Auto-select an input file: use first track from spkr_1
spkr1_dir = f'{DATASET_PATH}/spkr_1'
available_wavs = sorted([f for f in os.listdir(spkr1_dir) if f.endswith('.wav')])
INPUT_WAV = os.path.join(spkr1_dir, available_wavs[0])
print(f"Input: {INPUT_WAV}")

N_OVERLAP = 4
OUTPUT_DIR = f'{DRIVE_ROOT}/tt_vae_gan/outputs/{MODEL_NAME}_ep{INFER_EPOCH}_G{TRG_ID+1}'
os.makedirs(OUTPUT_DIR, exist_ok=True)


def sliding_window_inference(wav_path):
    """Full audio timbre transfer with sliding window."""
    sample = preprocess_wav(wav_path)
    spect_src = melspectrogram(sample)

    # Pad for consistent overlap
    spect_src = np.pad(spect_src, ((0, 0), (IMG_WIDTH, IMG_WIDTH)), 'constant')
    spect_trg = np.zeros_like(spect_src)

    length = spect_src.shape[1]
    hop = IMG_WIDTH // N_OVERLAP

    with torch.no_grad():
        for i in tqdm(range(0, length, hop), desc='Inference'):
            x = i + IMG_WIDTH
            if x <= length:
                S = spect_src[:, i:x]
            else:
                S = spect_src[:, i:]
                S = np.pad(S, ((0, 0), (0, x - length)), 'constant')

            S_tensor = torch.from_numpy(S).float().view(1, CHANNELS, IMG_HEIGHT, IMG_WIDTH)
            # Rescale [0,1] → [-1,1] to match training data range
            S_tensor = S_tensor * 2.0 - 1.0
            if cuda:
                S_tensor = S_tensor.cuda()

            mu, Z = encoder(S_tensor)
            fake = G[TRG_ID](Z)
            T = fake.squeeze().cpu().numpy()
            # Rescale Tanh output [-1,1] → [0,1]
            T = (T + 1.0) / 2.0

            for j in range(0, IMG_WIDTH, hop):
                y = j + hop
                if i + y > length:
                    break
                spect_trg[:, i+j:i+y] += T[:, j:y] / N_OVERLAP

    # Remove padding
    spect_src = spect_src[:, IMG_WIDTH:-IMG_WIDTH]
    spect_trg = spect_trg[:, IMG_WIDTH:-IMG_WIDTH]

    return spect_src, np.clip(spect_trg, 0, 1), sample


spect_src, spect_trg, original_wav = sliding_window_inference(INPUT_WAV)

# Reconstruct with Griffin-Lim
print('Reconstructing with Griffin-Lim...')
wav_transferred = reconstruct_waveform(spect_trg)

# Save
fname = Path(INPUT_WAV).stem
sf.write(f'{OUTPUT_DIR}/{fname}_original.wav', original_wav, SAMPLE_RATE)
sf.write(f'{OUTPUT_DIR}/{fname}_transferred_griffinlim.wav', wav_transferred, SAMPLE_RATE)
np.save(f'{OUTPUT_DIR}/{fname}_mel_src.npy', spect_src)
np.save(f'{OUTPUT_DIR}/{fname}_mel_trg.npy', spect_trg)

print(f'\n✓ Saved to {OUTPUT_DIR}/')
print(f'  Original: {fname}_original.wav')
print(f'  Transferred (Griffin-Lim): {fname}_transferred_griffinlim.wav')
print(f'  Mel arrays: {fname}_mel_src.npy, {fname}_mel_trg.npy')
if USE_SYNTHETIC:
    print(f'\n  Note: This is synthetic data — audio quality is not meaningful.')
    print(f'  The test validates that the full pipeline runs end-to-end. ✓')

## 10. Visualize Transfer Results

<!-- §37-stub -->
## Cell 13 — TODO short title

**What this does.** TODO

**Inputs.** TODO

**Outputs.** TODO

**Action required.** TODO (or _none_)

**Runtime.** TODO (seconds / minutes / hours)


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

axes[0].imshow(spect_src, aspect='auto', origin='lower', cmap='magma')
axes[0].set_title('Source Mel Spectrogram')
axes[0].set_ylabel('Mel Bins')

axes[1].imshow(spect_trg, aspect='auto', origin='lower', cmap='magma')
axes[1].set_title('Transferred Mel Spectrogram')
axes[1].set_ylabel('Mel Bins')
axes[1].set_xlabel('Frames')

plt.suptitle(f'Timbre Transfer: {MODEL_NAME} → G{TRG_ID+1}', fontsize=14)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/{fname}_comparison.png', dpi=150)
plt.show()

## 11. BigVGAN Vocoder (Optional — Better Quality)

Replace Griffin-Lim with BigVGAN for much better audio quality.
BigVGAN was trained on diverse audio including instruments.

**Note**: The transferred mel is in [0,1] normalization (model internal).
We need to convert it to the format BigVGAN expects.

<!-- §37-stub -->
## Cell 14 — TODO short title

**What this does.** TODO

**Inputs.** TODO

**Outputs.** TODO

**Action required.** TODO (or _none_)

**Runtime.** TODO (seconds / minutes / hours)


In [ ]:
# ─── BigVGAN vocoder ─────────────────────────────────────────────────────────
# Note: BigVGAN.from_pretrained() is broken in some environments.
# We use manual loading: download config + checkpoint via hf_hub_download,
# then construct BigVGAN(h) and load weights manually.
try:
    import bigvgan as bigvgan_module
    from bigvgan.meldataset import get_mel_spectrogram
    from bigvgan.env import AttrDict
    from huggingface_hub import hf_hub_download
    import json as _json

    BIGVGAN_MODEL_ID = 'nvidia/bigvgan_v2_22khz_80band_fmax8k_256x'

    print("Loading BigVGAN v2 (22kHz, 80-band) via manual loading...")
    config_path = hf_hub_download(repo_id=BIGVGAN_MODEL_ID, filename='config.json')
    ckpt_path = hf_hub_download(repo_id=BIGVGAN_MODEL_ID, filename='bigvgan_generator.pt')

    with open(config_path) as f:
        h = AttrDict(_json.load(f))

    bigvgan_model = bigvgan_module.BigVGAN(h)
    ckpt = torch.load(ckpt_path, map_location='cpu')
    bigvgan_model.load_state_dict(ckpt['generator'])
    bigvgan_model.remove_weight_norm()
    bigvgan_model.eval()
    if cuda:
        bigvgan_model = bigvgan_model.cuda()
    print(f"✓ BigVGAN loaded ({sum(p.numel() for p in bigvgan_model.parameters())/1e6:.0f}M params)")

    # Round-trip test: original audio → BigVGAN-native mel → BigVGAN → WAV
    wav_tensor = torch.from_numpy(original_wav).float().unsqueeze(0)
    if cuda:
        wav_tensor = wav_tensor.cuda()

    native_mel = get_mel_spectrogram(wav_tensor, bigvgan_model.h)

    with torch.inference_mode():
        wav_bigvgan = bigvgan_model(native_mel).squeeze().cpu().numpy()
        wav_bigvgan = np.clip(wav_bigvgan, -1, 1)

    sf.write(f'{OUTPUT_DIR}/{fname}_bigvgan_roundtrip.wav', wav_bigvgan, SAMPLE_RATE)
    print(f"✓ BigVGAN round-trip saved: {fname}_bigvgan_roundtrip.wav")

    # ─── Convert transferred mel [0,1] → BigVGAN vocoder output ──────────
    # Strategy: use Griffin-Lim to get a waveform from the transferred mel,
    # then re-analyze with BigVGAN's native mel computation for high-quality
    # neural vocoding. This preserves the timbre transfer while getting
    # BigVGAN-quality audio synthesis.
    #
    # Chain: transferred mel [0,1] → Griffin-Lim WAV → BigVGAN mel → BigVGAN → WAV
    print("Converting transferred mel → BigVGAN...")
    wav_gl = reconstruct_waveform(spect_trg, n_iter=64)  # higher iterations for quality
    wav_gl_tensor = torch.from_numpy(wav_gl).float().unsqueeze(0)
    if cuda:
        wav_gl_tensor = wav_gl_tensor.cuda()

    transferred_mel_bigvgan = get_mel_spectrogram(wav_gl_tensor, bigvgan_model.h)

    with torch.inference_mode():
        wav_transferred_bigvgan = bigvgan_model(transferred_mel_bigvgan).squeeze().cpu().numpy()
        wav_transferred_bigvgan = np.clip(wav_transferred_bigvgan, -1, 1)

    sf.write(f'{OUTPUT_DIR}/{fname}_transferred_bigvgan.wav',
             wav_transferred_bigvgan, SAMPLE_RATE)
    print(f"✓ BigVGAN transferred saved: {fname}_transferred_bigvgan.wav")

except ImportError:
    print("BigVGAN not installed. Using Griffin-Lim output only.")
    print("Install with: pip install bigvgan")

## 12. Listen & Compare

<!-- §37-stub -->
## Cell 15 — TODO short title

**What this does.** TODO

**Inputs.** TODO

**Outputs.** TODO

**Action required.** TODO (or _none_)

**Runtime.** TODO (seconds / minutes / hours)


In [ ]:
from IPython.display import Audio, display, HTML

display(HTML(f'<h3>Original</h3>'))
display(Audio(original_wav, rate=SAMPLE_RATE))

display(HTML(f'<h3>Transferred (Griffin-Lim)</h3>'))
display(Audio(wav_transferred, rate=SAMPLE_RATE))

try:
    display(HTML(f'<h3>BigVGAN Round-Trip (reference quality)</h3>'))
    display(Audio(wav_bigvgan, rate=SAMPLE_RATE))
    display(HTML(f'<h3>Transferred (BigVGAN)</h3>'))
    display(Audio(wav_transferred_bigvgan, rate=SAMPLE_RATE))
except:
    pass

print(f'\nAll outputs saved to: {OUTPUT_DIR}/')

## 13. Export Model for Local Pipeline

Copy trained checkpoints from Drive to your local project:
```
models/tt_vae_gan/saved_models/pipeline_urmp/
├── encoder_499.pth
├── G1_499.pth
└── G2_499.pth
```

Then run locally:
```bash
python -m models.tt_vae_gan.pipeline_inference \
    --input_dir MusicProjectData/artist/album/song/processed_data/mels/ \
    --model_name pipeline_urmp --epoch 499 --trg_id 2 \
    --img_height 80 --output_dir transferred_output/
```

<!-- §37-stub -->
## Cell 16 — TODO short title

**What this does.** TODO

**Inputs.** TODO

**Outputs.** TODO

**Action required.** TODO (or _none_)

**Runtime.** TODO (seconds / minutes / hours)


In [ ]:
# List saved checkpoints
print(f"Checkpoints in {SAVE_DIR}/:")
if os.path.exists(SAVE_DIR):
    files = sorted(os.listdir(SAVE_DIR))
    for f in files:
        size_mb = os.path.getsize(os.path.join(SAVE_DIR, f)) / 1e6
        print(f"  {f} ({size_mb:.1f} MB)")
else:
    print("  No checkpoints found (run training first)")